# Lab 8C: First-Order Wumpus World Knowledge Base

## Learning goals
By the end of this lab, students can:
- translate Wumpus World rules into compact first-order schemas;
- see how `Adjacent`, `Pit`, `Wumpus`, `Breeze`, and `Stench` interact in a finite model;
- perform finite-model checking over possible Wumpus worlds;
- watch percepts shrink the set of models and create entailed conclusions such as `Safe([2,2])`.

## Chapter 8 connection
Chapter 8 motivates FOL by contrasting one propositional rule per square with the compact structured rule: "squares adjacent to pits are breezy." This lab turns that idea into a visual model checker.


In [ ]:
# Run this cell first.

from itertools import product
from functools import lru_cache
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False
    print("ipywidgets is not installed. Static step output will be shown instead.")


## 1. FOL-style Wumpus rules

For every square `s`:

- `Breeze(s) ⇔ ∃p Adjacent(p,s) ∧ Pit(p)`
- `Stench(s) ⇔ ∃w Adjacent(w,s) ∧ Wumpus(w)`
- `Safe(s) ⇔ ¬Pit(s) ∧ ¬Wumpus(s)`

The Python code below evaluates these rules over a finite 4 × 4 grid by enumerating possible worlds.


In [ ]:
N = 4
SQUARES = [(x, y) for y in range(1, N + 1) for x in range(1, N + 1)]
START = (1, 1)
IDX = {sq: i for i, sq in enumerate(SQUARES)}
SQUARE_BY_IDX = {i: sq for sq, i in IDX.items()}
PIT_ALLOWED = [sq for sq in SQUARES if sq != START]
PIT_BIT = {sq: i for i, sq in enumerate(PIT_ALLOWED)}


def adjacent(square):
    x, y = square
    candidates = [(x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)]
    return [s for s in candidates if 1 <= s[0] <= N and 1 <= s[1] <= N]

NEIGHBORS = {sq: adjacent(sq) for sq in SQUARES}
NEIGHBOR_PIT_MASK = {}
for sq in SQUARES:
    mask = 0
    for nb in NEIGHBORS[sq]:
        if nb in PIT_BIT:
            mask |= 1 << PIT_BIT[nb]
    NEIGHBOR_PIT_MASK[sq] = mask

WUMPUS_ALLOWED = [sq for sq in SQUARES if sq != START]


def pit_at(pit_mask, sq):
    return sq in PIT_BIT and bool(pit_mask & (1 << PIT_BIT[sq]))


def breeze_at(pit_mask, sq):
    return bool(pit_mask & NEIGHBOR_PIT_MASK[sq])


def stench_at(wumpus_sq, sq):
    return wumpus_sq in NEIGHBORS[sq]


def safe_at(pit_mask, wumpus_sq, sq):
    return not pit_at(pit_mask, sq) and wumpus_sq != sq

print(f"Grid squares: {len(SQUARES)}")
print(f"Possible pit assignments: {2 ** len(PIT_ALLOWED):,}")
print(f"Possible wumpus locations: {len(WUMPUS_ALLOWED)}")
print(f"Total possible worlds before percepts: {2 ** len(PIT_ALLOWED) * len(WUMPUS_ALLOWED):,}")


In [ ]:
OBSERVATION_STEPS = [
    {"at": (1, 1), "breeze": False, "stench": False, "note": "Start: no breeze, no stench."},
    {"at": (2, 1), "breeze": True,  "stench": False, "note": "At [2,1]: breeze, no stench."},
    {"at": (1, 2), "breeze": False, "stench": True,  "note": "At [1,2]: stench, no breeze."},
    {"at": (2, 2), "breeze": False, "stench": False, "note": "At [2,2]: no breeze, no stench."},
    {"at": (2, 3), "breeze": True,  "stench": True,  "note": "At [2,3]: breeze and stench; in the textbook story, this square has glitter."},
]


def world_satisfies_observations(pit_mask, wumpus_sq, observations):
    # The start square is known safe.
    if pit_at(pit_mask, START) or wumpus_sq == START:
        return False
    for obs in observations:
        sq = obs["at"]
        if breeze_at(pit_mask, sq) != obs["breeze"]:
            return False
        if stench_at(wumpus_sq, sq) != obs["stench"]:
            return False
        # A visited square cannot have killed the agent.
        if not safe_at(pit_mask, wumpus_sq, sq):
            return False
    return True

@lru_cache(maxsize=None)
def consistent_worlds(step_count):
    observations = OBSERVATION_STEPS[:step_count]
    worlds = []
    for pit_mask in range(2 ** len(PIT_ALLOWED)):
        for wumpus_sq in WUMPUS_ALLOWED:
            if world_satisfies_observations(pit_mask, wumpus_sq, observations):
                worlds.append((pit_mask, wumpus_sq))
    return tuple(worlds)


def summarize_worlds(step_count):
    worlds = consistent_worlds(step_count)
    if not worlds:
        return {}, {}, {}
    pit_prob = {sq: sum(pit_at(pm, sq) for pm, w in worlds) / len(worlds) for sq in SQUARES}
    wumpus_prob = {sq: sum(w == sq for pm, w in worlds) / len(worlds) for sq in SQUARES}
    safe_entailed = {sq: all(safe_at(pm, w, sq) for pm, w in worlds) for sq in SQUARES}
    pit_entailed = {sq: all(pit_at(pm, sq) for pm, w in worlds) for sq in SQUARES}
    wumpus_entailed = {sq: all(w == sq for pm, w in worlds) for sq in SQUARES}
    return (pit_prob, wumpus_prob, {"safe": safe_entailed, "pit": pit_entailed, "wumpus": wumpus_entailed})

for i in range(len(OBSERVATION_STEPS) + 1):
    print(f"After {i} observation(s): {len(consistent_worlds(i)):,} consistent worlds")


In [ ]:
def draw_wumpus_belief(step_count=3):
    observations = OBSERVATION_STEPS[:step_count]
    pit_prob, wumpus_prob, entailed = summarize_worlds(step_count)
    worlds = consistent_worlds(step_count)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_xlim(0, N)
    ax.set_ylim(0, N)
    ax.set_aspect("equal")
    ax.set_xticks(range(N + 1))
    ax.set_yticks(range(N + 1))
    ax.grid(True)
    ax.set_title(f"Wumpus belief after {step_count} percept(s): {len(worlds):,} models remain")
    ax.set_xticklabels([])
    ax.set_yticklabels([])

    visited = {obs["at"] for obs in observations}
    obs_by_square = {obs["at"]: obs for obs in observations}

    for sq in SQUARES:
        x, y = sq
        cx, cy = x - 0.5, y - 0.5
        text = [f"[{x},{y}]"]
        if sq == START:
            text.append("START")
        if sq in visited:
            text.append("V")
            obs = obs_by_square[sq]
            if obs["breeze"]: text.append("B")
            if obs["stench"]: text.append("S")
            if not obs["breeze"] and not obs["stench"]: text.append("no B/S")
        if worlds:
            if entailed["wumpus"][sq]:
                text.append("W!")
            elif wumpus_prob[sq] > 0:
                text.append(f"W? {wumpus_prob[sq]:.2f}")
            if entailed["pit"][sq]:
                text.append("P!")
            elif pit_prob[sq] > 0:
                text.append(f"P? {pit_prob[sq]:.2f}")
            if entailed["safe"][sq]:
                text.append("OK")
        ax.text(cx, cy, "\n".join(text), ha="center", va="center", fontsize=10)

    ax.text(0, -0.25,
            "Rule schemas used: Breeze(s) iff some adjacent Pit; Stench(s) iff adjacent Wumpus; Safe(s) iff no Pit and no Wumpus.",
            ha="left", va="top", fontsize=9)
    plt.show()

    rows = "".join(f"<li><b>{obs['at']}</b>: breeze={obs['breeze']}, stench={obs['stench']} — {obs['note']}</li>" for obs in observations)
    display(HTML(f"<h3>Percepts told to the KB</h3><ol>{rows}</ol>"))


draw_wumpus_belief(3)


In [ ]:
if WIDGETS_AVAILABLE:
    slider = widgets.IntSlider(value=3, min=0, max=len(OBSERVATION_STEPS), step=1, description="percepts")
    out = widgets.Output()
    def update(change=None):
        with out:
            clear_output(wait=True)
            draw_wumpus_belief(slider.value)
    slider.observe(update, names="value")
    display(slider, out)
    update()
else:
    for step in range(len(OBSERVATION_STEPS) + 1):
        draw_wumpus_belief(step)


## 2. Why Chapter 8 wants first-order logic

A propositional Wumpus KB needs separate symbols like `B_1_1`, `P_1_2`, and one rule per square. FOL uses variables and a relation such as `Adjacent(p,s)`, so the rule stays one sentence while the number of squares changes.


In [ ]:
def propositional_breeze_rules_for_grid(n):
    # One biconditional for each square. Each biconditional mentions all legal adjacent pit propositions.
    count_rules = n * n
    count_literal_mentions = 0
    for x in range(1, n + 1):
        for y in range(1, n + 1):
            deg = len([(x+1,y), (x-1,y), (x,y+1), (x,y-1)])
            # Remove neighbors outside the board.
            deg = len([(a,b) for (a,b) in [(x+1,y), (x-1,y), (x,y+1), (x,y-1)] if 1 <= a <= n and 1 <= b <= n])
            count_literal_mentions += 1 + deg
    return count_rules, count_literal_mentions

sizes = list(range(2, 31))
prop_counts = [propositional_breeze_rules_for_grid(n)[1] for n in sizes]
fol_counts = [1 for _ in sizes]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sizes, prop_counts, marker="o", label="propositional literal mentions")
ax.plot(sizes, fol_counts, marker="s", label="one FOL schema")
ax.set_xlabel("Grid width n for an n × n Wumpus world")
ax.set_ylabel("Approximate rule-size count")
ax.set_title("FOL represents repeated structure compactly")
ax.legend()
plt.show()

print("FOL schema:")
print("∀s Breeze(s) ⇔ ∃p Adjacent(p,s) ∧ Pit(p)")
print("For a 30x30 grid, propositional literal mentions just for breeze rules:", prop_counts[-1])


## Student practice

1. Add a new percept after visiting `[3,2]`. Does it entail any new `OK`, `P!`, or `W!` conclusions?
2. Modify the Wumpus rule so diagonal adjacency counts. How does that change model counts?
3. Explain why the FOL rule is still one schema when the grid grows, while the propositional version grows with the number of squares.
